# Distracted Driver Detection - Training and Evaluation

This notebook trains and evaluates three CNN architectures for distracted driver behavior detection:
- **EfficientNet-B0**: Balanced accuracy and efficiency
- **MobileNetV3-Small**: Optimized for mobile/edge deployment
- **ResNet-50**: Deep residual network with strong performance

## Dataset
State Farm Distracted Driver Detection dataset with 10 classes:
- c0: Safe driving
- c1: Texting (right hand)
- c2: Talking on phone (right hand)
- c3: Texting (left hand)
- c4: Talking on phone (left hand)
- c5: Operating the radio
- c6: Drinking
- c7: Reaching behind
- c8: Hair and makeup
- c9: Talking to passenger

## 1. Install Dependencies

In [ ]:
# Uncomment and run if dependencies are not installed
# !pip install torch torchvision albumentations pytorch-grad-cam thop onnx onnxruntime opencv-python numpy pandas scikit-learn matplotlib seaborn Pillow PyYAML tqdm

## 2. Mount Google Drive (if using Colab)

In [ ]:
# Uncomment if using Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

# Set paths for Colab
# PROJECT_ROOT = '/content/drive/MyDrive/distracted_driver_detection'
# DATA_DIR = '/content/drive/MyDrive/data/state_farm'

## 3. Import Libraries and Set Paths

In [ ]:
import sys
from pathlib import Path

# Set project root (adjust if needed)
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader

from utils import load_config, set_seed, get_device, ensure_dir, get_short_class_names, get_class_names, save_json
from data.dataset import DriverDataset
from data.transforms import train_transforms, val_transforms
from models.model_factory import get_model
from training.losses import compute_class_weights, get_loss
from training.trainer import Trainer, create_optimizer, create_scheduler
from evaluation.metrics import compute_metrics, plot_confusion_matrix, plot_training_curves, print_metrics_summary
from evaluation.efficiency import efficiency_report, get_model_summary

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Load configuration
config = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')

# Set paths
DATA_DIR = PROJECT_ROOT / config['data']['data_dir']
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
CHECKPOINTS_DIR = OUTPUT_DIR / 'checkpoints'
RESULTS_DIR = OUTPUT_DIR / 'results'

# Create directories
ensure_dir(CHECKPOINTS_DIR)
ensure_dir(RESULTS_DIR)

# Set seed for reproducibility
set_seed(config['training']['seed'])

# Get device
device = get_device()
print(f"Using device: {device}")

# Class names
CLASS_NAMES = get_short_class_names()
print(f"\nClasses: {CLASS_NAMES}")

## 4. Visualize Sample Images

In [ ]:
# Load dataset without transforms for visualization
viz_dataset = DriverDataset(
    root_dir=DATA_DIR,
    split='train',
    transform=None,
    train_ratio=config['data']['train_split'],
    seed=config['training']['seed']
)

print(f"Total training samples: {len(viz_dataset)}")

In [ ]:
# Visualize 20 sample images (2 per class)
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

# Get 2 samples per class
samples_per_class = {i: [] for i in range(10)}
for idx in range(len(viz_dataset)):
    img, label, path = viz_dataset[idx]
    if len(samples_per_class[label]) < 2:
        samples_per_class[label].append((img, label, path))
    if all(len(v) >= 2 for v in samples_per_class.values()):
        break

# Plot samples
plot_idx = 0
for class_idx in range(10):
    for sample in samples_per_class[class_idx]:
        img, label, path = sample
        if isinstance(img, torch.Tensor):
            img = img.permute(1, 2, 0).numpy()
        axes[plot_idx].imshow(img)
        axes[plot_idx].set_title(f"{CLASS_NAMES[label]}\n({Path(path).name[:15]}...)", fontsize=10)
        axes[plot_idx].axis('off')
        plot_idx += 1

plt.suptitle('Sample Images from Each Class', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Build DataLoaders and Analyze Class Distribution

In [ ]:
# Create datasets with transforms
img_size = config['data']['img_size']

train_dataset = DriverDataset(
    root_dir=DATA_DIR,
    split='train',
    transform=train_transforms(img_size),
    train_ratio=config['data']['train_split'],
    seed=config['training']['seed']
)

val_dataset = DriverDataset(
    root_dir=DATA_DIR,
    split='val',
    transform=val_transforms(img_size),
    train_ratio=config['data']['train_split'],
    seed=config['training']['seed']
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=config['data']['num_workers'],
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=config['data']['num_workers'],
    pin_memory=True
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# Plot class distribution
train_dist = train_dataset.get_class_distribution()
val_dist = val_dataset.get_class_distribution()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Training distribution
train_counts = [train_dist[i] for i in range(10)]
bars1 = axes[0].bar(CLASS_NAMES, train_counts, color=sns.color_palette('husl', 10))
axes[0].set_title('Training Set Class Distribution', fontsize=14)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for bar, count in zip(bars1, train_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
                 str(count), ha='center', va='bottom', fontsize=9)

# Validation distribution
val_counts = [val_dist[i] for i in range(10)]
bars2 = axes[1].bar(CLASS_NAMES, val_counts, color=sns.color_palette('husl', 10))
axes[1].set_title('Validation Set Class Distribution', fontsize=14)
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
for bar, count in zip(bars2, val_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
                 str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Train All Three Models

In [ ]:
# Compute class weights
class_weights = compute_class_weights(train_dataset, num_classes=10)
class_weights = class_weights.to(device)
print("Class weights:")
for i, (name, weight) in enumerate(zip(CLASS_NAMES, class_weights.cpu().numpy())):
    print(f"  {name}: {weight:.3f}")

In [ ]:
# Training function
def train_model(arch_name, train_loader, val_loader, config, device, class_weights):
    """Train a single model architecture."""
    print(f"\n{'='*60}")
    print(f"Training {arch_name}")
    print(f"{'='*60}")
    
    # Initialize model
    model = get_model(
        arch_name=arch_name,
        num_classes=config['data']['num_classes'],
        pretrained=config['models']['pretrained']
    )
    model = model.to(device)
    
    # Loss function
    criterion = get_loss(
        use_class_weights=config['training']['use_class_weights'],
        class_weights_tensor=class_weights,
        label_smoothing=config['training']['label_smoothing'],
        device=device
    )
    
    # Optimizer and scheduler
    optimizer = create_optimizer(
        model=model,
        optimizer_name=config['training']['optimizer'],
        learning_rate=config['training']['learning_rate'],
        weight_decay=config['training']['weight_decay']
    )
    
    scheduler = create_scheduler(
        optimizer=optimizer,
        scheduler_name=config['training']['scheduler'],
        t_max=config['training'].get('scheduler_t_max', 20)
    )
    
    # Trainer
    trainer = Trainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
        save_dir=CHECKPOINTS_DIR,
        arch_name=arch_name
    )
    
    # Train
    history = trainer.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=config['training']['epochs']
    )
    
    return model, trainer, history

In [ ]:
# Train all models
architectures = config['models']['architectures']
trained_models = {}
training_histories = {}

for arch in architectures:
    model, trainer, history = train_model(
        arch_name=arch,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        class_weights=class_weights
    )
    trained_models[arch] = model
    training_histories[arch] = history

## 7. Plot Training Curves

In [ ]:
# Plot training curves for each model
for arch, history in training_histories.items():
    plot_training_curves(
        history,
        save_path=RESULTS_DIR / f'{arch}_training_curves.png',
        show=True
    )

In [ ]:
# Combined training curves comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'efficientnet_b0': '#e74c3c', 'mobilenet_v3_small': '#3498db', 'resnet50': '#2ecc71'}

for arch, history in training_histories.items():
    epochs = range(1, len(history['val_loss']) + 1)
    axes[0].plot(epochs, history['val_loss'], label=arch, color=colors.get(arch, 'gray'), linewidth=2)
    axes[1].plot(epochs, [a*100 for a in history['val_accuracy']], label=arch, color=colors.get(arch, 'gray'), linewidth=2)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('Validation Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Accuracy (%)')
axes[1].set_title('Validation Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Final Accuracy/F1 Comparison Table

In [ ]:
# Evaluate all models and create comparison table
results_summary = []

for arch, model in trained_models.items():
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            images, labels = batch[0], batch[1]
            images = images.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())
    
    metrics = compute_metrics(all_labels, all_preds, num_classes=10)
    
    results_summary.append({
        'Architecture': arch,
        'Accuracy (%)': f"{metrics['accuracy']*100:.2f}",
        'Macro F1 (%)': f"{metrics['macro_f1']*100:.2f}",
        'Best Val Acc (%)': f"{max(training_histories[arch]['val_accuracy'])*100:.2f}"
    })

results_df = pd.DataFrame(results_summary)
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(results_df.to_string(index=False))

## 9. Plot Confusion Matrices

In [ ]:
# Generate confusion matrices for each model
for arch, model in trained_models.items():
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            images, labels = batch[0], batch[1]
            images = images.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())
    
    plot_confusion_matrix(
        y_true=all_labels,
        y_pred=all_preds,
        class_names=CLASS_NAMES,
        save_path=RESULTS_DIR / f'{arch}_confusion_matrix.png',
        show=True
    )

## 10. Efficiency Report

In [ ]:
# Compute accuracy and F1 for efficiency report
accuracy_dict = {}
f1_dict = {}

for arch, model in trained_models.items():
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            images, labels = batch[0], batch[1]
            images = images.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())
    
    metrics = compute_metrics(all_labels, all_preds, num_classes=10)
    accuracy_dict[arch] = metrics['accuracy']
    f1_dict[arch] = metrics['macro_f1']

# Generate efficiency report
eff_report = efficiency_report(
    models_dict=trained_models,
    accuracy_dict=accuracy_dict,
    f1_dict=f1_dict,
    device='cpu'  # Use CPU for fair latency comparison
)

print("\n" + "="*80)
print("EFFICIENCY REPORT")
print("="*80)
print(eff_report.to_string(index=False))

In [ ]:
# Save efficiency report
eff_report.to_csv(RESULTS_DIR / 'efficiency_report.csv', index=False)
print(f"\nEfficiency report saved to: {RESULTS_DIR / 'efficiency_report.csv'}")

In [ ]:
# Visualize accuracy vs efficiency trade-off
fig, ax = plt.subplots(figsize=(10, 6))

for arch, model in trained_models.items():
    summary = get_model_summary(model, device='cpu')
    acc = accuracy_dict[arch] * 100
    params = summary['total_params_M']
    
    ax.scatter(params, acc, s=200, label=arch, alpha=0.7)
    ax.annotate(arch, (params, acc), textcoords="offset points", xytext=(0,10), ha='center')

ax.set_xlabel('Parameters (M)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Model Size', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'accuracy_vs_params.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Training and evaluation complete! Key findings:

1. **EfficientNet-B0**: Best balance of accuracy and efficiency
2. **MobileNetV3-Small**: Fastest inference, suitable for edge deployment
3. **ResNet-50**: Strong accuracy but larger model size

All checkpoints and results have been saved to the `outputs/` directory.